# Notebook 05: Intent Classifier Baseline

## Objective

Build and evaluate the first EdoFarm AI intent classification models.

The experiments will compare:

1. English-only training.
2. Bilingual English-Edo training.
3. English-to-Edo cross-language transfer.

## Model architecture

TF-IDF Vectorization + Logistic Regression.

We will explore word-level and character-level text features.

## Evaluation

- Accuracy
- Precision
- Recall
- Macro F1-score
- Weighted F1-score
- Per-class classification report
- Confusion matrix

## Important methodology

- Reuse the existing train, validation, and test splits.
- Do not move records between splits.
- English and Edo versions of the same question remain in the same split.
- Use validation data for model selection.
- Reserve the test set for final evaluation.
- Treat machine-generated Edo translations as provisional.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

def find_project_root(start_path=Path.cwd()):
    for path in [start_path, *start_path.parents]:
        if (
            (path / "data" / "processed").is_dir()
            and (path / "notebooks").is_dir()
        ):
            return path

    raise FileNotFoundError(
        "Could not locate the EdoFarm AI project root."
    )

ROOT = find_project_root()
PROCESSED_DIR = ROOT / "data" / "processed"

print("Project root:", ROOT)
print("Processed directory:", PROCESSED_DIR)

Project root: c:\Users\XXMUGENX\3D Objects\edofarm-ai
Processed directory: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed


## 1. Load the bilingual classifier datasets

Load the previously exported training, validation, and test datasets.

We will verify their sizes, language representation, intent labels,
and record pairing before training any models.

In [2]:
split_paths = {
    "train": PROCESSED_DIR / (
        "classifier_bilingual_train_provisional_v1.csv"
    ),
    "validation": PROCESSED_DIR / (
        "classifier_bilingual_validation_provisional_v1.csv"
    ),
    "test": PROCESSED_DIR / (
        "classifier_bilingual_test_provisional_v1.csv"
    )
}

datasets = {}

required_columns = [
    "example_id",
    "record_id",
    "split_group_id",
    "language",
    "question_text",
    "primary_intent",
    "split"
]

for split_name, path in split_paths.items():

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)

    missing = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{split_name} is missing columns: {missing}"
        )

    if set(df["split"].dropna().unique()) != {split_name}:
        raise ValueError(
            f"Unexpected split assignment in {split_name}."
        )

    if df[required_columns].isna().any().any():
        raise ValueError(
            f"Missing required values in {split_name}."
        )

    datasets[split_name] = df

    print(
        f"{split_name.upper()}: "
        f"{len(df)} examples | "
        f"{df['record_id'].nunique()} underlying records"
    )

print("\nAll three datasets loaded successfully.")

TRAIN: 392 examples | 196 underlying records
VALIDATION: 98 examples | 49 underlying records
TEST: 122 examples | 61 underlying records

All three datasets loaded successfully.


## 2. Inspect the datasets

Confirm:

- Each split contains English and Edo examples.
- Every underlying record has both language versions.
- All eight intent classes are represented.
- No underlying record occurs in multiple splits.
- There are no duplicate example IDs.

In [3]:
for split_name, df in datasets.items():

    print(f"\n{'=' * 45}")
    print(split_name.upper())
    print("=" * 45)

    print("\nLanguage distribution:")
    print(df["language"].value_counts())

    print("\nIntent distribution:")
    print(df["primary_intent"].value_counts())

    print("\nTotal examples:", len(df))
    print("Underlying records:", df["record_id"].nunique())


# Combine splits for cross-split checks.
all_data = pd.concat(
    datasets.values(),
    ignore_index=True
)

# Expected dataset size.
if len(all_data) != 612:
    raise ValueError(
        f"Expected 612 examples, received {len(all_data)}."
    )

# Each record must have exactly two examples.
pair_counts = all_data.groupby("record_id").size()

if not (pair_counts == 2).all():
    raise ValueError(
        "Some underlying records do not have exactly two examples."
    )

# Each record must contain both languages.
expected_languages = {"english", "edo_bini"}

language_pairs = (
    all_data.groupby("record_id")["language"]
    .agg(set)
)

if not language_pairs.apply(
    lambda values: values == expected_languages
).all():
    raise ValueError("Incomplete language pairs detected.")

# No record can occur in multiple splits.
record_split_counts = (
    all_data.groupby("record_id")["split"]
    .nunique()
)

if (record_split_counts > 1).any():
    raise ValueError("Record leakage detected between splits.")

# Unique example IDs.
if all_data["example_id"].duplicated().any():
    raise ValueError("Duplicate example IDs detected.")

print("\nMODEL DATA VALIDATION PASSED")
print("-----------------------------")
print("Total examples:", len(all_data))
print("Unique underlying records:", all_data["record_id"].nunique())
print("Language pairs: complete")
print("Cross-split record leakage: none")
print("Duplicate example IDs: none")
print(
    "Intent classes:",
    sorted(all_data["primary_intent"].unique())
)


TRAIN

Language distribution:
language
english     196
edo_bini    196
Name: count, dtype: int64

Intent distribution:
primary_intent
out_of_scope              96
soil_management           56
fertilizer_application    52
disease_management        50
harvesting                44
irrigation                40
pest_management           32
crop_cultivation          22
Name: count, dtype: int64

Total examples: 392
Underlying records: 196

VALIDATION

Language distribution:
language
english     49
edo_bini    49
Name: count, dtype: int64

Intent distribution:
primary_intent
out_of_scope              24
disease_management        14
soil_management           14
fertilizer_application    12
irrigation                10
harvesting                10
pest_management            8
crop_cultivation           6
Name: count, dtype: int64

Total examples: 98
Underlying records: 49

TEST

Language distribution:
language
english     61
edo_bini    61
Name: count, dtype: int64

Intent distribution:
primar

## 3. Prepare training and validation subsets

Create the following datasets:

- English training examples.
- Bilingual training examples.
- English validation examples.
- Edo validation examples.

The test dataset will remain untouched until final model evaluation.

No new data splitting will be performed.

In [4]:
train_df = datasets["train"].copy()
validation_df = datasets["validation"].copy()

# Training subsets
train_english = train_df[
    train_df["language"] == "english"
].copy()

train_bilingual = train_df.copy()

# Validation subsets
validation_english = validation_df[
    validation_df["language"] == "english"
].copy()

validation_edo = validation_df[
    validation_df["language"] == "edo_bini"
].copy()

# Confirm the experiment sizes.
experiment_sizes = {
    "English training": len(train_english),
    "Bilingual training": len(train_bilingual),
    "English validation": len(validation_english),
    "Edo validation": len(validation_edo)
}

print("EXPERIMENT DATA SIZES")
print("---------------------")

for name, size in experiment_sizes.items():
    print(f"{name}: {size}")

# Confirm that all training classes are represented.
all_classes = set(all_data["primary_intent"].unique())

for name, frame in {
    "English training": train_english,
    "Bilingual training": train_bilingual,
    "English validation": validation_english,
    "Edo validation": validation_edo
}.items():

    missing_classes = (
        all_classes - set(frame["primary_intent"].unique())
    )

    if missing_classes:
        raise ValueError(
            f"{name} is missing classes: {missing_classes}"
        )

print("\nAll experiment subsets contain all eight intent classes.")

EXPERIMENT DATA SIZES
---------------------
English training: 196
Bilingual training: 392
English validation: 49
Edo validation: 49

All experiment subsets contain all eight intent classes.


## 4. Feature Extraction Experiments

Three feature representations will be evaluated:

F1: Word-level TF-IDF
    Unigrams and bigrams.

F2: Character-level TF-IDF
    Character n-grams of length 2–5.

F3: Combined word and character TF-IDF
    FeatureUnion of F1 and F2.

Two training conditions will be compared:

A. English-only training.
B. Bilingual English-Edo training.

This produces six trained model configurations.

Each model will be evaluated separately on English and Edo
validation data, producing twelve validation evaluations.

BPE tokenization and Edo-specific lemmatization are deferred
to future implementation phases.

All feature vocabularies and IDF weights must be fitted
using training data only.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

train_df = datasets["train"].copy()
validation_df = datasets["validation"].copy()

training_sets = {
    "english_only": train_df[
        train_df["language"] == "english"
    ].copy(),

    "bilingual": train_df.copy()
}

validation_sets = {
    "english": validation_df[
        validation_df["language"] == "english"
    ].copy(),

    "edo_bini": validation_df[
        validation_df["language"] == "edo_bini"
    ].copy()
}

print("Training sizes:")
for name, frame in training_sets.items():
    print(name, len(frame))

print("\nValidation sizes:")
for name, frame in validation_sets.items():
    print(name, len(frame))

Training sizes:
english_only 196
bilingual 392

Validation sizes:
english 49
edo_bini 49


In [6]:
def create_feature_extractor(feature_name):

    if feature_name == "word_ngrams":

        return TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=1,
            sublinear_tf=True,
            max_features=20000
        )

    elif feature_name == "character_ngrams":

        return TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=1,
            sublinear_tf=True,
            max_features=30000
        )

    elif feature_name == "word_character_combined":

        return FeatureUnion([
            ("word", TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                min_df=1,
                sublinear_tf=True,
                max_features=20000
            )),
            ("character", TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(2, 5),
                min_df=1,
                sublinear_tf=True,
                max_features=30000
            ))
        ])

    else:
        raise ValueError(
            f"Unknown feature configuration: {feature_name}"
        )


FEATURE_CONFIGURATIONS = [
    "word_ngrams",
    "character_ngrams",
    "word_character_combined"
]

print("Feature configurations ready.")

Feature configurations ready.


In [7]:
def create_intent_classifier(feature_name):

    return Pipeline([
        (
            "features",
            create_feature_extractor(feature_name)
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=3000,
                C=1.0,
                random_state=42
            )
        )
    ])

In [8]:
trained_models = {}
experiment_results = []
validation_predictions = {}

LABELS = sorted(
    train_df["primary_intent"].unique()
)

for strategy, training_frame in training_sets.items():

    print(f"\nTraining strategy: {strategy}")

    for feature_name in FEATURE_CONFIGURATIONS:

        print(f"Feature: {feature_name}")

        model = create_intent_classifier(feature_name)

        model.fit(
            training_frame["question_text"],
            training_frame["primary_intent"]
        )

        trained_models[
            (strategy, feature_name)
        ] = model

        for language, validation_frame in validation_sets.items():

            predictions = model.predict(
                validation_frame["question_text"]
            )

            actual = validation_frame["primary_intent"]

            results = {
                "training_strategy": strategy,
                "feature_representation": feature_name,
                "evaluation_language": language,
                "training_examples": len(training_frame),
                "validation_examples": len(validation_frame),
                "accuracy": accuracy_score(actual, predictions),
                "precision_macro": precision_score(
                    actual, predictions,
                    labels=LABELS,
                    average="macro",
                    zero_division=0
                ),
                "recall_macro": recall_score(
                    actual, predictions,
                    labels=LABELS,
                    average="macro",
                    zero_division=0
                ),
                "f1_macro": f1_score(
                    actual, predictions,
                    labels=LABELS,
                    average="macro",
                    zero_division=0
                ),
                "f1_weighted": f1_score(
                    actual, predictions,
                    labels=LABELS,
                    average="weighted",
                    zero_division=0
                )
            }

            experiment_results.append(results)

            validation_predictions[
                (strategy, feature_name, language)
            ] = predictions

print("\nSix models trained.")
print("Twelve validation evaluations completed.")


Training strategy: english_only
Feature: word_ngrams
Feature: character_ngrams
Feature: word_character_combined

Training strategy: bilingual
Feature: word_ngrams
Feature: character_ngrams
Feature: word_character_combined

Six models trained.
Twelve validation evaluations completed.


In [9]:
validation_results_df = pd.DataFrame(
    experiment_results
)

validation_results_df = validation_results_df.sort_values(
    ["evaluation_language", "f1_macro"],
    ascending=[True, False]
)

display(validation_results_df.round(4))

print("\nMACRO F1 COMPARISON")

display(
    validation_results_df.pivot(
        index=[
            "training_strategy",
            "feature_representation"
        ],
        columns="evaluation_language",
        values="f1_macro"
    ).round(4)
)

,training_strategy,feature_representation,evaluation_language,training_examples,validation_examples,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted
11,bilingual,word_character_combined,edo_bini,392,49,0.8163,0.7250,0.7756,0.7364,0.7896
9,bilingual,character_ngrams,edo_bini,392,49,0.7959,0.7326,0.7577,0.7224,0.7898
7,bilingual,word_ngrams,edo_bini,392,49,0.7755,0.6954,0.7622,0.7125,0.7497
3,english_only,character_ngrams,edo_bini,196,49,0.5714,0.5990,0.5021,0.5145,0.5426
5,english_only,word_character_combined,edo_bini,196,49,0.5510,0.5608,0.4530,0.4611,0.5118
1,english_only,word_ngrams,edo_bini,196,49,0.5714,0.5073,0.4708,0.4542,0.5186
4,english_only,word_character_combined,english,196,49,0.8571,0.8810,0.8351,0.8307,0.8508
10,bilingual,word_character_combined,english,392,49,0.8367,0.8778,0.8101,0.8187,0.8296
8,bilingual,character_ngrams,english,392,49,0.8367,0.8546,0.7967,0.8000,0.8321
2,english_only,character_ngrams,english,196,49,0.8367,0.8109,0.7967,0.7979,0.8327



MACRO F1 COMPARISON


evaluation_language                        edo_bini  english
training_strategy feature_representation                    
bilingual         character_ngrams           0.7224   0.8000
                  word_character_combined    0.7364   0.8187
                  word_ngrams                0.7125   0.7331
english_only      character_ngrams           0.5145   0.7979
                  word_character_combined    0.4611   0.8307
                  word_ngrams                0.4542   0.6949

## 5. Per-Intent Evaluation of the Selected Candidate

Selected configuration:

- Training strategy: bilingual
- Feature representation: combined word and character TF-IDF
- Classifier: Logistic Regression

Evaluate its per-intent precision, recall, F1-score and support
on both English and Edo validation datasets.

Also generate confusion matrices to identify intents that
the classifier frequently confuses.

The test dataset remains untouched.

In [10]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

selected_model = trained_models[
    ("bilingual", "word_character_combined")
]

diagnostics_dir = (
    PROCESSED_DIR / "model_diagnostics"
)

diagnostics_dir.mkdir(
    parents=True,
    exist_ok=True
)

candidate_reports = {}
candidate_confusion_matrices = {}

for language, validation_frame in validation_sets.items():

    print("\n" + "=" * 55)
    print(f"{language.upper()} VALIDATION DIAGNOSTICS")
    print("=" * 55)

    y_true = validation_frame["primary_intent"]

    y_pred = validation_predictions[
        ("bilingual", "word_character_combined", language)
    ]

    # Per-class evaluation.
    report = classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report).T

    class_report = report_df.loc[
        LABELS + ["macro avg", "weighted avg"],
        ["precision", "recall", "f1-score", "support"]
    ]

    candidate_reports[language] = class_report

    display(class_report.round(4))

    # Export classification report.
    report_path = (
        diagnostics_dir /
        f"candidate_{language}_classification_report.csv"
    )

    class_report.to_csv(report_path)

    # Confusion matrix.
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=LABELS
    )

    cm_df = pd.DataFrame(
        cm,
        index=LABELS,
        columns=LABELS
    )

    candidate_confusion_matrices[language] = cm_df

    print("\nConfusion matrix:")
    display(cm_df)

    cm_path = (
        diagnostics_dir /
        f"candidate_{language}_confusion_matrix.csv"
    )

    cm_df.to_csv(cm_path)

    print("\nSaved:", report_path)
    print("Saved:", cm_path)

print("\nPer-intent diagnostics completed.")


ENGLISH VALIDATION DIAGNOSTICS


,precision,recall,f1-score,support
crop_cultivation,1.0000,0.3333,0.5000,3.0
disease_management,0.8333,0.7143,0.7692,7.0
fertilizer_application,1.0000,1.0000,1.0000,6.0
harvesting,0.8000,0.8000,0.8000,5.0
irrigation,1.0000,0.8000,0.8889,5.0
out_of_scope,0.7143,0.8333,0.7692,12.0
pest_management,0.8000,1.0000,0.8889,4.0
soil_management,0.8750,1.0000,0.9333,7.0
macro avg,0.8778,0.8101,0.8187,49.0
weighted avg,0.8516,0.8367,0.8296,49.0



Confusion matrix:


,crop_cultivation,disease_management,fertilizer_application,harvesting,irrigation,out_of_scope,pest_management,soil_management
crop_cultivation,1,0,0,1,0,0,0,1
disease_management,0,5,0,0,0,2,0,0
fertilizer_application,0,0,6,0,0,0,0,0
harvesting,0,0,0,4,0,1,0,0
irrigation,0,0,0,0,4,1,0,0
out_of_scope,0,1,0,0,0,10,1,0
pest_management,0,0,0,0,0,0,4,0
soil_management,0,0,0,0,0,0,0,7



Saved: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\model_diagnostics\candidate_english_classification_report.csv
Saved: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\model_diagnostics\candidate_english_confusion_matrix.csv

EDO_BINI VALIDATION DIAGNOSTICS


,precision,recall,f1-score,support
crop_cultivation,0.0000,0.0000,0.0000,3.0
disease_management,0.8000,0.5714,0.6667,7.0
fertilizer_application,1.0000,1.0000,1.0000,6.0
harvesting,0.6250,1.0000,0.7692,5.0
irrigation,1.0000,0.8000,0.8889,5.0
out_of_scope,0.8333,0.8333,0.8333,12.0
pest_management,0.6667,1.0000,0.8000,4.0
soil_management,0.8750,1.0000,0.9333,7.0
macro avg,0.7250,0.7756,0.7364,49.0
weighted avg,0.7861,0.8163,0.7896,49.0



Confusion matrix:


,crop_cultivation,disease_management,fertilizer_application,harvesting,irrigation,out_of_scope,pest_management,soil_management
crop_cultivation,0,0,0,2,0,0,0,1
disease_management,0,4,0,1,0,1,1,0
fertilizer_application,0,0,6,0,0,0,0,0
harvesting,0,0,0,5,0,0,0,0
irrigation,0,0,0,0,4,1,0,0
out_of_scope,0,1,0,0,0,10,1,0
pest_management,0,0,0,0,0,0,4,0
soil_management,0,0,0,0,0,0,0,7



Saved: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\model_diagnostics\candidate_edo_bini_classification_report.csv
Saved: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\model_diagnostics\candidate_edo_bini_confusion_matrix.csv

Per-intent diagnostics completed.


## 6. Misclassification and Translation Error Analysis

Inspect the validation examples incorrectly classified by the
selected bilingual word-character model.

For each error, examine:

- Original intent.
- Predicted intent.
- English question.
- Edo (Bini) question.
- Language in which the misclassification occurred.

The objective is to identify possible translation problems,
ambiguous labels and weaknesses in the training data.

No labels will be changed automatically.

The validation set will remain unchanged during this investigation.

In [11]:
error_frames = []

for language, validation_frame in validation_sets.items():

    frame = validation_frame.copy().reset_index(drop=True)

    predictions = validation_predictions[
        ("bilingual", "word_character_combined", language)
    ]

    frame["predicted_intent"] = predictions
    frame["evaluation_language"] = language

    errors = frame[
        frame["primary_intent"] != frame["predicted_intent"]
    ].copy()

    error_frames.append(errors)

    print(f"\n{language.upper()} MISCLASSIFICATIONS")
    print("-" * 45)

    print("Errors:", len(errors))
    print("Total:", len(frame))

    display(
        errors[
            [
                "record_id",
                "primary_intent",
                "predicted_intent",
                "question_text"
            ]
        ]
    )


all_errors = pd.concat(
    error_frames,
    ignore_index=True
)

print("\nTOTAL MISCLASSIFICATIONS")
print("------------------------")
print("Total errors:", len(all_errors))
print("Unique affected records:", all_errors["record_id"].nunique())


ENGLISH MISCLASSIFICATIONS
---------------------------------------------
Errors: 8
Total: 49


,record_id,primary_intent,predicted_intent,question_text
5,AQA_00695,irrigation,out_of_scope,can waterlogging cause yellowing in beans plants?
6,AQA_00756,harvesting,out_of_scope,how can improver storage cause shriveling and ...
12,AQA_01421,crop_cultivation,soil_management,What type of soil is best for growing apple tr...
23,FCN_01634,crop_cultivation,harvesting,When is the best time to plant maize in April ...
34,FCN_08903,disease_management,out_of_scope,How can I identify crop diseases in my farm in...
36,FCN_09291,disease_management,out_of_scope,What causes tomato plants to die just before h...
40,FCN_00442,out_of_scope,pest_management,What natural methods can I use to control pest...
46,FCN_10581,out_of_scope,disease_management,What are effective methods to prevent diseases...



EDO_BINI MISCLASSIFICATIONS
---------------------------------------------
Errors: 9
Total: 49


,record_id,primary_intent,predicted_intent,question_text
0,AQA_00009,disease_management,out_of_scope,Emianmwen vbe na nẹ whiteflies sẹtin kpe vbe e...
5,AQA_00695,irrigation,out_of_scope,Amẹ nọ mudia vbe otọ (waterlogging) sẹtin zẹ n...
12,AQA_01421,crop_cultivation,soil_management,De otọ nọ maan sẹ nẹ ikọ erhan apple?
23,FCN_01634,crop_cultivation,harvesting,De ẹghẹ nọ maan sẹ nẹ a na kọ oka vbe April vb...
28,FCN_06011,crop_cultivation,harvesting,De ẹgbẹ otọ nọ maan sẹ nẹ ikọ melon vbe Onitsh...
34,FCN_08903,disease_management,pest_management,De vbene I sẹtin ya rhiẹre ma emianmwen emwin-...
36,FCN_09291,disease_management,harvesting,Vbọ zẹ erhan itomato na wu kevbe nẹ a kọ ẹre k...
40,FCN_00442,out_of_scope,pest_management,De odẹ otọ (natural methods) nẹ I sẹtin lọghọ ...
46,FCN_10581,out_of_scope,disease_management,De odẹ nọ ru iruẹmwin esẹ nẹ a ya kọkọ emianmw...



TOTAL MISCLASSIFICATIONS
------------------------
Total errors: 17
Unique affected records: 10


In [12]:
# Identify all underlying records with at least one error.
error_ids = set(
    all_errors["record_id"].astype(str)
)

validation_pairs = datasets["validation"].copy()

# Get the English and Edo questions side by side.
paired_questions = (
    validation_pairs
    .pivot(
        index="record_id",
        columns="language",
        values="question_text"
    )
    .reset_index()
)

# Recover the original intent.
intent_lookup = (
    validation_pairs
    .groupby("record_id")["primary_intent"]
    .first()
    .reset_index()
)

paired_questions = paired_questions.merge(
    intent_lookup,
    on="record_id",
    how="left"
)

# Summarize which languages produced errors.
all_errors["error_description"] = (
    all_errors["evaluation_language"]
    + ": "
    + all_errors["primary_intent"]
    + " → "
    + all_errors["predicted_intent"]
)

error_summary = (
    all_errors.groupby("record_id")["error_description"]
    .agg(" | ".join)
    .reset_index()
)

paired_errors = paired_questions[
    paired_questions["record_id"].astype(str).isin(error_ids)
].merge(
    error_summary,
    on="record_id",
    how="left"
)

print("MISCLASSIFIED ENGLISH-EDO QUESTION PAIRS")
print("----------------------------------------")

display(
    paired_errors[
        [
            "record_id",
            "primary_intent",
            "english",
            "edo_bini",
            "error_description"
        ]
    ]
)

MISCLASSIFIED ENGLISH-EDO QUESTION PAIRS
----------------------------------------


,record_id,primary_intent,english,edo_bini,error_description
0,AQA_00009,disease_management,which diseases can be spread by whiteflies?,Emianmwen vbe na nẹ whiteflies sẹtin kpe vbe e...,edo_bini: disease_management → out_of_scope
1,AQA_00695,irrigation,can waterlogging cause yellowing in beans plants?,Amẹ nọ mudia vbe otọ (waterlogging) sẹtin zẹ n...,english: irrigation → out_of_scope | edo_bini:...
2,AQA_00756,harvesting,how can improver storage cause shriveling and ...,De vbene isẹ-emwin nọ i maan sẹtin ya zẹ nẹ ik...,english: harvesting → out_of_scope
3,AQA_01421,crop_cultivation,What type of soil is best for growing apple tr...,De otọ nọ maan sẹ nẹ ikọ erhan apple?,english: crop_cultivation → soil_management | ...
4,FCN_00442,out_of_scope,What natural methods can I use to control pest...,De odẹ otọ (natural methods) nẹ I sẹtin lọghọ ...,english: out_of_scope → pest_management | edo_...
5,FCN_01634,crop_cultivation,When is the best time to plant maize in April ...,De ẹghẹ nọ maan sẹ nẹ a na kọ oka vbe April vb...,english: crop_cultivation → harvesting | edo_b...
6,FCN_06011,crop_cultivation,What type of soil is best for planting melon i...,De ẹgbẹ otọ nọ maan sẹ nẹ ikọ melon vbe Onitsh...,edo_bini: crop_cultivation → harvesting
7,FCN_08903,disease_management,How can I identify crop diseases in my farm in...,De vbene I sẹtin ya rhiẹre ma emianmwen emwin-...,english: disease_management → out_of_scope | e...
8,FCN_09291,disease_management,What causes tomato plants to die just before h...,Vbọ zẹ erhan itomato na wu kevbe nẹ a kọ ẹre k...,english: disease_management → out_of_scope | e...
9,FCN_10581,out_of_scope,What are effective methods to prevent diseases...,De odẹ nọ ru iruẹmwin esẹ nẹ a ya kọkọ emianmw...,english: out_of_scope → disease_management | e...


## 7. Targeted Validation Data Quality Review

Ten underlying records were involved in classification errors.

We will prepare two separate files:

1. A blind review file containing source questions, answers and
   provisional translations, without original or predicted labels.
2. An internal diagnostic file containing original labels and
   model predictions.

The blind review will investigate potential annotation errors
and translation problems.

No records will be automatically relabelled.

Original dataset records and validation assignments must remain
unchanged until a documented review decision is made.

In [13]:
# Load the full provisional bilingual dataset.
FULL_DATA_PATH = (
    PROCESSED_DIR /
    "edofarm_bilingual_dataset_provisional_v1.csv"
)

full_bilingual_df = pd.read_csv(FULL_DATA_PATH)

# Identify the unique records involved in validation errors.
error_ids = sorted(
    all_errors["record_id"].astype(str).unique()
)

print("Affected underlying records:", len(error_ids))
print(error_ids)

# Retrieve their complete source records.
review_records = full_bilingual_df[
    full_bilingual_df["record_id"].astype(str).isin(error_ids)
].copy()

# Context fields for blind review.
preferred_columns = [
    "record_id",
    "source_dataset",
    "source_record_id",
    "crop",
    "country",
    "state_region",
    "local_area",
    "english_question",
    "english_answer",
    "edo_question_final",
    "edo_answer_final",
    "translation_method",
    "translation_notes",
    "translation_review_status",
    "meaning_preservation_status"
]

blind_columns = [
    col for col in preferred_columns
    if col in review_records.columns
]

blind_review_df = review_records[blind_columns].copy()

# Add blank human review fields.
blind_review_df["team_intent_review"] = ""
blind_review_df["proposed_intent"] = ""
blind_review_df["scope_review"] = ""
blind_review_df["translation_review"] = "pending_speaker_review"
blind_review_df["reviewer_comments"] = ""

# Export blind review file.
review_output = (
    PROCESSED_DIR /
    "classifier_validation_manual_review_v1.csv"
)

blind_review_df.to_csv(
    review_output,
    index=False
)

print("\nBLIND REVIEW FILE CREATED")
print("-------------------------")
print("Records:", len(blind_review_df))
print("Saved:", review_output)

Affected underlying records: 10
['AQA_00009', 'AQA_00695', 'AQA_00756', 'AQA_01421', 'FCN_00442', 'FCN_01634', 'FCN_06011', 'FCN_08903', 'FCN_09291', 'FCN_10581']

BLIND REVIEW FILE CREATED
-------------------------
Records: 10
Saved: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\classifier_validation_manual_review_v1.csv


## 8. Export Internal Error Diagnostics

This file preserves the model predictions and original labels.

It is for internal analysis only and should not be supplied
to reviewers conducting blind annotation.

In [14]:
internal_diagnostics = paired_errors.copy()

# Keep original classifications and error descriptions.
internal_output = (
    PROCESSED_DIR /
    "classifier_validation_error_diagnostics_internal_v1.csv"
)

internal_diagnostics.to_csv(
    internal_output,
    index=False
)

print("INTERNAL DIAGNOSTICS EXPORTED")
print("-----------------------------")
print("Records:", len(internal_diagnostics))
print("Saved:", internal_output)

INTERNAL DIAGNOSTICS EXPORTED
-----------------------------
Records: 10
Saved: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\classifier_validation_error_diagnostics_internal_v1.csv


## 9. Final Holdout Test Evaluation

Evaluate the selected model on the previously reserved test dataset.

Selected model:
- Bilingual training
- Combined word and character TF-IDF
- Logistic Regression

Evaluation will be performed separately for:
1. English questions.
2. Edo (Bini) questions.

Metrics:
- Accuracy
- Macro precision
- Macro recall
- Macro F1
- Weighted F1
- Per-intent classification report
- Confusion matrix

The test dataset must not be used for further model fitting,
hyperparameter tuning or feature selection.

Validation results and test results will be reported separately.

The Edo test evaluation measures performance on provisional
machine-generated translations, not independently verified
natural Bini questions.

In [15]:
test_df = datasets["test"].copy()

test_english = test_df[
    test_df["language"] == "english"
].copy()

test_edo = test_df[
    test_df["language"] == "edo_bini"
].copy()

print("FINAL TEST DATASET")
print("------------------")

print("Total examples:", len(test_df))
print("Unique records:", test_df["record_id"].nunique())
print("English examples:", len(test_english))
print("Edo examples:", len(test_edo))

# Verify expected sizes.
assert len(test_df) == 122
assert test_df["record_id"].nunique() == 61
assert len(test_english) == 61
assert len(test_edo) == 61

# Verify no record overlap with training or validation.
train_ids = set(datasets["train"]["record_id"])
validation_ids = set(datasets["validation"]["record_id"])
test_ids = set(test_df["record_id"])

assert not (train_ids & test_ids)
assert not (validation_ids & test_ids)

# Verify both language versions of each record are present.
assert (
    test_df.groupby("record_id")["language"].nunique() == 2
).all()

print("\nTEST DATA VALIDATION PASSED")
print("No training/test record overlap.")
print("No validation/test record overlap.")
print("Complete English-Edo pairs.")

FINAL TEST DATASET
------------------
Total examples: 122
Unique records: 61
English examples: 61
Edo examples: 61

TEST DATA VALIDATION PASSED
No training/test record overlap.
No validation/test record overlap.
Complete English-Edo pairs.


In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Retrieve the already-trained selected model.
final_model = trained_models[
    ("bilingual", "word_character_combined")
]

test_sets = {
    "english": test_english,
    "edo_bini": test_edo
}

final_test_results = []
final_test_reports = {}
final_test_matrices = {}
final_test_predictions = {}

for language, frame in test_sets.items():

    print("\n" + "=" * 55)
    print(f"FINAL TEST: {language.upper()}")
    print("=" * 55)

    y_true = frame["primary_intent"]
    X_test = frame["question_text"]

    # Prediction only. No fitting or tuning.
    y_pred = final_model.predict(X_test)

    final_test_predictions[language] = y_pred

    # Aggregate metrics.
    metrics = {
        "model": "bilingual_word_character",
        "evaluation_language": language,
        "test_examples": len(frame),

        "accuracy": accuracy_score(
            y_true, y_pred
        ),

        "precision_macro": precision_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        ),

        "recall_macro": recall_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        ),

        "f1_macro": f1_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        ),

        "f1_weighted": f1_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="weighted",
            zero_division=0
        )
    }

    final_test_results.append(metrics)

    # Per-intent report.
    report = classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report).T

    final_test_reports[language] = report_df

    # Confusion matrix.
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=LABELS
    )

    cm_df = pd.DataFrame(
        cm,
        index=LABELS,
        columns=LABELS
    )

    final_test_matrices[language] = cm_df

    print("\nClassification report:")
    display(report_df.round(4))

    print("\nConfusion matrix:")
    display(cm_df)


final_test_results_df = pd.DataFrame(
    final_test_results
)

print("\nFINAL TEST PERFORMANCE")
print("----------------------")

display(
    final_test_results_df.round(4)
)


FINAL TEST: ENGLISH

Classification report:


,precision,recall,f1-score,support
crop_cultivation,0.0000,0.0000,0.0000,3.0000
disease_management,0.7500,0.7500,0.7500,8.0000
fertilizer_application,0.7000,0.8750,0.7778,8.0000
harvesting,0.8571,1.0000,0.9231,6.0000
irrigation,1.0000,0.4286,0.6000,7.0000
out_of_scope,0.6471,0.7333,0.6875,15.0000
pest_management,0.7143,1.0000,0.8333,5.0000
soil_management,0.7778,0.7778,0.7778,9.0000
accuracy,0.7377,0.7377,0.7377,0.7377
macro avg,0.6808,0.6956,0.6687,61.0000



Confusion matrix:


,crop_cultivation,disease_management,fertilizer_application,harvesting,irrigation,out_of_scope,pest_management,soil_management
crop_cultivation,0,0,1,0,0,1,1,0
disease_management,0,6,0,0,0,2,0,0
fertilizer_application,0,0,7,0,0,1,0,0
harvesting,0,0,0,6,0,0,0,0
irrigation,0,0,0,1,3,1,0,2
out_of_scope,0,2,1,0,0,11,1,0
pest_management,0,0,0,0,0,0,5,0
soil_management,0,0,1,0,0,1,0,7



FINAL TEST: EDO_BINI

Classification report:


,precision,recall,f1-score,support
crop_cultivation,0.0000,0.0000,0.0000,3.0000
disease_management,0.8000,1.0000,0.8889,8.0000
fertilizer_application,0.8571,0.7500,0.8000,8.0000
harvesting,0.6667,1.0000,0.8000,6.0000
irrigation,1.0000,0.5714,0.7273,7.0000
out_of_scope,0.8667,0.8667,0.8667,15.0000
pest_management,1.0000,1.0000,1.0000,5.0000
soil_management,0.8000,0.8889,0.8421,9.0000
accuracy,0.8197,0.8197,0.8197,0.8197
macro avg,0.7488,0.7596,0.7406,61.0000



Confusion matrix:


,crop_cultivation,disease_management,fertilizer_application,harvesting,irrigation,out_of_scope,pest_management,soil_management
crop_cultivation,0,1,1,0,0,1,0,0
disease_management,0,8,0,0,0,0,0,0
fertilizer_application,0,0,6,0,0,1,0,1
harvesting,0,0,0,6,0,0,0,0
irrigation,0,0,0,2,4,0,0,1
out_of_scope,0,1,0,1,0,13,0,0
pest_management,0,0,0,0,0,0,5,0
soil_management,1,0,0,0,0,0,0,8



FINAL TEST PERFORMANCE
----------------------


,model,evaluation_language,test_examples,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted
0,bilingual_word_character,english,61,0.7377,0.6808,0.6956,0.6687,0.7121
1,bilingual_word_character,edo_bini,61,0.8197,0.7488,0.7596,0.7406,0.8030


In [17]:
FINAL_TEST_DIR = (
    PROCESSED_DIR /
    "model_diagnostics" /
    "final_test"
)

FINAL_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Overall metrics.
metrics_path = (
    FINAL_TEST_DIR /
    "final_test_metrics_v1.csv"
)

final_test_results_df.to_csv(
    metrics_path,
    index=False
)

# Per-language reports and confusion matrices.
for language in ["english", "edo_bini"]:

    final_test_reports[language].to_csv(
        FINAL_TEST_DIR /
        f"final_test_{language}_classification_report.csv"
    )

    final_test_matrices[language].to_csv(
        FINAL_TEST_DIR /
        f"final_test_{language}_confusion_matrix.csv"
    )

# Save individual predictions for traceability.
prediction_frames = []

for language, frame in test_sets.items():

    output = frame[
        [
            "example_id",
            "record_id",
            "language",
            "question_text",
            "primary_intent"
        ]
    ].copy()

    output["predicted_intent"] = (
        final_test_predictions[language]
    )

    output["correct"] = (
        output["primary_intent"]
        == output["predicted_intent"]
    )

    prediction_frames.append(output)

test_prediction_df = pd.concat(
    prediction_frames,
    ignore_index=True
)

prediction_path = (
    FINAL_TEST_DIR /
    "final_test_predictions_v1.csv"
)

test_prediction_df.to_csv(
    prediction_path,
    index=False
)

print("FINAL TEST EXPORT COMPLETED")
print("---------------------------")
print("Metrics:", metrics_path)
print("Predictions:", prediction_path)
print("Reports and confusion matrices:", FINAL_TEST_DIR)

FINAL TEST EXPORT COMPLETED
---------------------------
Metrics: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\model_diagnostics\final_test\final_test_metrics_v1.csv
Predictions: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\model_diagnostics\final_test\final_test_predictions_v1.csv
Reports and confusion matrices: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\model_diagnostics\final_test


## 10. Export the Selected Intent Classifier

Export the selected fitted model as a single joblib artifact.

The artifact includes:
- Word-level TF-IDF vectorizer.
- Character-level TF-IDF vectorizer.
- Combined feature extraction pipeline.
- Fitted Logistic Regression classifier.

No retraining is performed during export.

In [18]:
import joblib
import json
from datetime import date

MODEL_DIR = (
    ROOT / "models" / "intent_classifier" / "v1"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_PATH = (
    MODEL_DIR / "edofarm_intent_classifier_v1.joblib"
)

# Export the already-fitted pipeline.
joblib.dump(
    final_model,
    MODEL_PATH,
    compress=3
)

print("MODEL EXPORT COMPLETED")
print("-----------------------")
print("Model:", MODEL_PATH)
print("Model size:", round(MODEL_PATH.stat().st_size / 1024, 2), "KB")

MODEL EXPORT COMPLETED
-----------------------
Model: c:\Users\XXMUGENX\3D Objects\edofarm-ai\models\intent_classifier\v1\edofarm_intent_classifier_v1.joblib
Model size: 539.88 KB


## 11. Export Model Metadata

Record the model architecture, training configuration,
dataset sizes, evaluation results and known limitations.

This metadata will support reproducibility and future
model-card documentation.

In [19]:
# Select validation results for the chosen model.
selected_validation = validation_results_df[
    (
        validation_results_df["training_strategy"]
        == "bilingual"
    )
    &
    (
        validation_results_df["feature_representation"]
        == "word_character_combined"
    )
].copy()

if len(selected_validation) != 2:
    raise ValueError(
        "Expected two selected-model validation results."
    )

# Prepare JSON metadata.
metadata = {
    "model_name": "EdoFarm AI Intent Classifier",
    "model_version": "1.0.0",
    "model_status": "experimental_baseline",
    "created_date": date.today().isoformat(),

    "task": "Agricultural query intent classification",

    "supported_intents": LABELS,

    "architecture": {
        "training_strategy": "bilingual",
        "feature_extraction": [
            "word_level_tfidf",
            "character_level_tfidf"
        ],
        "word_ngrams": [1, 2],
        "character_ngrams": [2, 5],
        "word_max_features": 20000,
        "character_max_features": 30000,
        "classifier": "LogisticRegression",
        "class_weight": "balanced",
        "max_iter": 3000,
        "C": 1.0,
        "random_state": 42
    },

    "training_data": {
        "language_examples": 392,
        "underlying_questions": 196,
        "english_examples": 196,
        "edo_examples": 196,
        "source_file": "classifier_bilingual_train_provisional_v1.csv"
    },

    "validation_data": {
        "underlying_questions": 49,
        "language_examples": 98,
        "results": selected_validation.to_dict(
            orient="records"
        )
    },

    "test_data": {
        "underlying_questions": 61,
        "language_examples": 122,
        "results": final_test_results_df.to_dict(
            orient="records"
        )
    },

    "translation_status": {
        "edo_translations": "machine_generated_provisional",
        "human_translation_review": "pending",
        "native_speaker_validation": "not_performed"
    },

    "limitations": [
        "Small dataset with only 196 underlying training questions.",
        "Test evaluation contains 61 underlying questions.",
        "Edo test questions are machine translations.",
        "No independent validation by Edo speakers has been performed.",
        "Crop cultivation achieved zero F1 on both test languages.",
        "English test macro F1 is lower than English validation macro F1.",
        "Model probabilities have not been calibrated.",
        "Agricultural answer accuracy is not established by the classifier."
    ],

    "intended_use": (
        "Experimental intent classification for the EdoFarm AI "
        "crop-production advisory prototype."
    )
}

def json_safe(value):
    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if pd.isna(value):
        return None

    raise TypeError(
        f"Cannot serialize {type(value)}"
    )

METADATA_PATH = (
    MODEL_DIR / "model_metadata_v1.json"
)

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(
        metadata,
        file,
        indent=4,
        ensure_ascii=False,
        default=json_safe
    )

print("METADATA EXPORTED")
print("-----------------")
print(METADATA_PATH)

METADATA EXPORTED
-----------------
c:\Users\XXMUGENX\3D Objects\edofarm-ai\models\intent_classifier\v1\model_metadata_v1.json


In [20]:
# Load the exported model.
reloaded_model = joblib.load(MODEL_PATH)

# Run a small inference test.
sample_questions = test_english[
    "question_text"
].head(5)

original_predictions = final_model.predict(
    sample_questions
)

reloaded_predictions = reloaded_model.predict(
    sample_questions
)

if not np.array_equal(
    original_predictions,
    reloaded_predictions
):
    raise RuntimeError(
        "Exported model predictions differ from the original."
    )

print("MODEL EXPORT VERIFICATION PASSED")
print("--------------------------------")
print("Reload successful.")
print("Predictions identical:", True)

display(
    pd.DataFrame({
        "question": sample_questions.values,
        "original_prediction": original_predictions,
        "reloaded_prediction": reloaded_predictions
    })
)

MODEL EXPORT VERIFICATION PASSED
--------------------------------
Reload successful.
Predictions identical: True


,question,original_prediction,reloaded_prediction
0,List some of the soil management activites.,soil_management,soil_management
1,what is fertilization?,fertilizer_application,fertilizer_application
2,what happens during biological nitrogen fixation?,out_of_scope,out_of_scope
3,how does reliable rainfall improve crop yield.,soil_management,soil_management
4,name one demerits of single stem planting method.,out_of_scope,out_of_scope
